# Bimanual Ball Balance — Training Run
PPO via rsl-rl. G1 fixed-base humanoid, right arm only. Ball spawns at random position on tray each episode.

In [ ]:
# ── 1. Per-run config ── change this between runs ─────────────────────────
BRANCH         = 'pranav/training'
EXP_NAME       = 'ball_balance_v1'
N_ENVS         = 4096
MAX_ITERATIONS = 1000
LR             = 1e-4
ACTION_DELTA   = 0.3
RUN_DIR        = f'/kaggle/working/{EXP_NAME}'

In [ ]:
# ── 2. Clone / pull repo ──────────────────────────────────────────────────
from kaggle_secrets import UserSecretsClient
import subprocess, os

token = UserSecretsClient().get_secret('GITHUB_TOKEN')

if not os.path.exists('bimanual-project'):
    result = subprocess.run([
        'git', 'clone', '--branch', BRANCH,
        f'https://{token}@github.com/sharana-sabesan09/bimanual-project.git'
    ], capture_output=True, text=True)
    print(result.stdout)
    print(result.stderr)
else:
    result = subprocess.run(
        ['git', '-C', 'bimanual-project', 'pull'],
        capture_output=True, text=True
    )
    print(result.stdout)

commit_msg = subprocess.run(
    ['git', '-C', 'bimanual-project', 'log', '-1', '--pretty=%B'],
    capture_output=True, text=True
)
print('Latest commit:', commit_msg.stdout.strip())
print('Repo ready.')

In [ ]:
# ── 3. Install dependencies ───────────────────────────────────────────────
import subprocess
subprocess.run(['pip', 'install', 'genesis-world', '-q'], check=True)
subprocess.run(['pip', 'install', 'rsl-rl-lib>=5.0.0', '-q'], check=True)
subprocess.run(['pip', 'install', 'tensordict', 'tensorboard', '-q'], check=True)
print('Dependencies ready.')

In [ ]:
# ── 4. Setup paths ────────────────────────────────────────────────────────
import sys, os
sys.path.insert(0, '/kaggle/working/bimanual-project')
os.chdir('/kaggle/working/bimanual-project')
os.makedirs(RUN_DIR, exist_ok=True)
print(f'Working directory: {os.getcwd()}')
print(f'Run directory:     {RUN_DIR}')

In [ ]:
# ── 5. Verify GPU ─────────────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU:            {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
if torch.cuda.is_available():
    print(f'VRAM:           {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── 6. Sanity check — ALWAYS RUN BEFORE FULL TRAINING ─────────────────────
# Quick checks against the 2-hour-then-find-bug pattern.
import torch
from ball_balance_env import RIGHT_ARM_HOLD_POS, TRAY_SIZE, BALL_RADIUS

# action scaling: hold_pose + action * delta, clamped to limits
hold    = torch.tensor(RIGHT_ARM_HOLD_POS)
action  = torch.rand(4, 7) * 2 - 1
targets = hold + action * ACTION_DELTA
assert targets.shape == (4, 7), f'Bad target shape: {targets.shape}'
print(f'Action scaling OK: targets in [{targets.min():.3f}, {targets.max():.3f}] rad')

# ball spawn: offset must stay within tray bounds
max_xy = torch.tensor([TRAY_SIZE[0]/2 - BALL_RADIUS, TRAY_SIZE[1]/2 - BALL_RADIUS])
noise  = (torch.rand(4, 2) - 0.5) * 2 * max_xy
assert (noise.abs() <= max_xy + 1e-6).all(), 'Ball spawn offset exceeds tray bounds'
print(f'Ball spawn OK: max offset {noise.abs().max():.3f} m  (tray half-size {max_xy.tolist()})')

# reward math
ball_pos = torch.rand(4, 3)
goal_pos = torch.rand(4, 3)
ball_vel = torch.rand(4, 3)
xy_dist  = torch.norm(ball_pos[:, :2] - goal_pos[:, :2], dim=-1)
reward   = torch.exp(-3.0 * xy_dist) - 0.1 * torch.norm(ball_vel, dim=-1)
assert torch.all(torch.isfinite(reward)), 'Non-finite reward'
print(f'Reward math OK: range [{reward.min():.3f}, {reward.max():.3f}]')

print('Sanity check passed.')

In [ ]:
# ── 7. Full training run ──────────────────────────────────────────────────
import pickle
from pathlib import Path
import genesis as gs
from train_single_hand import BallBalanceVecEnv, get_train_cfg
from rsl_rl.runners import OnPolicyRunner

gs.init(backend=gs.gpu, precision='32', logging_level='warning')

cfg = get_train_cfg(EXP_NAME)
cfg['algorithm']['learning_rate'] = LR

log_dir = Path(RUN_DIR)
with open(log_dir / 'train_cfg.pkl', 'wb') as f:
    pickle.dump({'train': cfg, 'env': {'n_envs': N_ENVS, 'action_delta': ACTION_DELTA}}, f)

env    = BallBalanceVecEnv(n_envs=N_ENVS, show_viewer=False, action_delta=ACTION_DELTA)
runner = OnPolicyRunner(env, cfg, str(log_dir), device=gs.device)
runner.learn(num_learning_iterations=MAX_ITERATIONS, init_at_random_ep_len=True)

In [ ]:
# ── 8. Plot training curves ───────────────────────────────────────────────
import matplotlib.pyplot as plt
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

ea = EventAccumulator(RUN_DIR)
ea.Reload()

def extract(tag):
    events = ea.Scalars(tag)
    return [e.step for e in events], [e.value for e in events]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

steps, vals = extract('Train/mean_reward')
axes[0].plot(steps, vals)
axes[0].set_title('Mean Reward')
axes[0].set_xlabel('Iteration')
axes[0].grid(True)

steps, vals = extract('Train/mean_episode_length')
axes[1].plot(steps, vals, color='orange')
axes[1].axhline(500, color='gray', linestyle='--', label='max ep length')
axes[1].set_title('Mean Episode Length')
axes[1].set_xlabel('Iteration')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.savefig(f'{RUN_DIR}/training_curves.png', dpi=150)
plt.show()

_, rewards = extract('Train/mean_reward')
_, ep_lens = extract('Train/mean_episode_length')
print(f'Final reward:  {rewards[-1]:.2f}  (best {max(rewards):.2f})')
print(f'Final ep len:  {ep_lens[-1]:.1f}  (best {max(ep_lens):.1f} / 500)')

In [ ]:
# ── 9. Confirm outputs ────────────────────────────────────────────────────
import os
print(f'Files in {RUN_DIR}:')
for f in sorted(os.listdir(RUN_DIR)):
    size = os.path.getsize(f'{RUN_DIR}/{f}') / 1e6
    print(f'  {f} ({size:.1f} MB)')